# Ensambles sobre el dataset Iris

En este cuaderno se comparan varios métodos de ensamble aplicados al dataset clásico de Iris.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## Preparación de datos

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## Escenarios de ensamble

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "VotingSoft": VotingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
            ("knn", KNeighborsClassifier(n_neighbors=5)),
            ("svc", SVC(probability=True, kernel='rbf', C=10, gamma='scale', random_state=42))
        ],
        voting='soft'
    ),
    "Stacking": StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
            ("gb", GradientBoostingClassifier(random_state=42)),
            ("knn", KNeighborsClassifier(n_neighbors=7))
        ],
        final_estimator=LogisticRegression(max_iter=1000)
    )
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}: accuracy media={scores.mean():.3f} +/- {scores.std():.3f}")

## Entrenamiento final y reporte

In [ ]:
reports = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    reports[name] = classification_report(y_test, y_pred, output_dict=True)
    print(f"Reporte para {name}:
{classification_report(y_test, y_pred)}")

## Conclusiones

* Los modelos de votación suave y stacking suelen alcanzar precisiones superiores al 0.95 en Iris.
* Random Forest ofrece una alternativa robusta y rápida con poca sintonización.
* Combinar clasificadores diversos ayuda a reducir la varianza y mejorar la generalización.